# <b>Introdução</b>

O E-Commerce é um mercado que está em forte expansão no Brasil. Impulsionado pelo crescimento de marketplaces e acesso ampliado à internet e redes de entrega, esse mercado tem projeção de faturamento de <b>R$ 259 bilhões</b> em 2026. Apesar da crescente popularidade de lojas digitais, marketplaces como Olist são dependentes na adesão de clientes, e consequentemente, na satisfação de seus consumidores. O tempo de entrega é um dos pontos mais importantes para que lojas digitais consigam manter clientes ativos e conquistar novos compradores. Portanto, a intenção dessa análise é entender o efeito do tempo de entrega de uma compra sobre a avaliação deixada pelo cliente.

### <b>Pergunta norteadora</b>:
O atraso no tempo de entrega impacta diretamente a nota de avaliação deixada
pelo cliente?

<center>
<img src="imagens/logo_olist.png" alt="Olist logo" title="" width=1000 height=500/>
</center>

### Colunas das datasets
A fonte das datasets utilizadas nesse notebook (https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) possui várias tabelas diferentes contendo informações referentes a compras, clientes e vendedores envolvidos nessas compras. Para essa análise somente serão usadas as tabelas "olist_order_reviews_dataset.csv" e "olist_orders_dataset.csv", visto que essas bases de dados possuem informações relevantes para responder a pergunta norteadora. As colunas dessas tabelas contém informações como: a nota dada por um cliente, o comentário deixado por um cliente, o status de um pedido (entregue, enviado, etc.), a data de entrega estimada e a data de entrega efetiva.

### <b> 1. Importando bibliotecas e carregando bases de dados</b>

In [144]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime

# Path das bases
path_orders = r"archive/olist_orders_dataset.csv"
path_reviews = r"archive/olist_order_reviews_dataset.csv"

# Carregando tabelas
df_orders = pd.read_csv(path_orders, sep=None, engine='python')
df_reviews = pd.read_csv(path_reviews, sep=None, engine='python')

### <b>2. Tratamento dos dados</b>

In [145]:
# Juntando as duas tabelas
df = pd.merge(df_orders, df_reviews, how='outer', on='order_id')
display(df.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,97ca439bc427b48bc1cd7177abe71365,5.0,NaN,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,7b07bacd811c4117b742569b04ce3580,4.0,NaN,NaN,2017-05-13 00:00:00,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,0c5b33dea94867d1ac402749e5438e8b,5.0,NaN,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,2018-01-23 16:06:31
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,f4028d019cb58564807486a6aaf33817,4.0,NaN,NaN,2018-08-15 00:00:00,2018-08-15 16:39:01
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,940144190dcba6351888cafa43f3a3a5,5.0,NaN,Gostei pois veio no prazo determinado .,2017-03-02 00:00:00,2017-03-03 10:54:59


In [146]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99992 entries, 0 to 99991
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       99992 non-null  object 
 1   customer_id                    99992 non-null  object 
 2   order_status                   99992 non-null  object 
 3   order_purchase_timestamp       99992 non-null  object 
 4   order_approved_at              99831 non-null  object 
 5   order_delivered_carrier_date   98199 non-null  object 
 6   order_delivered_customer_date  97005 non-null  object 
 7   order_estimated_delivery_date  99992 non-null  object 
 8   review_id                      99224 non-null  object 
 9   review_score                   99224 non-null  float64
 10  review_comment_title           11568 non-null  object 
 11  review_comment_message         40977 non-null  object 
 12  review_creation_date           99224 non-null 

In [147]:
# Não queremos que o mesmo pedido apareça duas vezes
df = df[~df.order_id.duplicated()]

# Quantos valores nulos existem por coluna
nulos = pd.DataFrame({
    'nulos': df.isna().sum(),
    '%':  (df.isna().mean() * 100).round(1),
})
display(nulos)

,nulos,%
order_id,0,0.0
customer_id,0,0.0
order_status,0,0.0
order_purchase_timestamp,0,0.0
order_approved_at,160,0.2
order_delivered_carrier_date,1783,1.8
order_delivered_customer_date,2965,3.0
order_estimated_delivery_date,0,0.0
review_id,768,0.8
review_score,768,0.8


In [148]:
print(df.loc[df['order_status'] == 'delivered']['order_delivered_customer_date'].isna().sum())
print(df.loc[df['order_status'] == 'delivered']['review_score'].isna().sum())

8
646


Valores nulos em order_delivered_customer_date podem ser falhas na coleta de dados ou podem indicar que o pedido nunca chegou, então invés de remover valores nulos vamos criar a categoria para determinar atraso somento com pedidos que já foram entregues, e como só existem 8 pedidos que já foram entregues com order_delivered_customer_date, vamos removê-los. Valores nulos na coluna review_score também serão removidos, pois não temos como estimar a nota de um cliente.

In [149]:
# Garantir que as datas serão tratadas como datetime
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"], errors="coerce")
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"], errors="coerce")

df_delivered = df.loc[df['order_status'] == 'delivered']
df_delivered = df_delivered.dropna(subset=['order_delivered_customer_date', 'review_score']).reset_index()

df_delivered['atraso'] = (df_delivered["order_delivered_customer_date"] - df["order_estimated_delivery_date"])
display(df_delivered.head())

,index,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,atraso
0,0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,97ca439bc427b48bc1cd7177abe71365,5.0,NaN,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,2017-09-22 10:57:03,-9 days +23:43:48
1,1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,7b07bacd811c4117b742569b04ce3580,4.0,NaN,NaN,2017-05-13 00:00:00,2017-05-15 11:34:13,-3 days +16:04:24
2,2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,0c5b33dea94867d1ac402749e5438e8b,5.0,NaN,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,2018-01-23 16:06:31,-14 days +13:19:16
3,3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,f4028d019cb58564807486a6aaf33817,4.0,NaN,NaN,2018-08-15 00:00:00,2018-08-15 16:39:01,-6 days +13:32:39
4,4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,940144190dcba6351888cafa43f3a3a5,5.0,NaN,Gostei pois veio no prazo determinado .,2017-03-02 00:00:00,2017-03-03 10:54:59,-16 days +16:42:31
